In [36]:
import os
import mne

In [37]:
FILTERED_ROOT = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/processed/filtered"

EPOCH_ROOT = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/processed/epochs"

os.makedirs(EPOCH_ROOT, exist_ok=True)

In [38]:
TARGET_RUNS = [
    "R04",
    "R08",
    "R12"
]

In [39]:
def create_epochs_from_filtered(raw):

    events, event_id_raw = mne.events_from_annotations(
        raw,
        verbose=False
    )

    required = {"T0", "T1", "T2"}

    if not required.issubset(event_id_raw.keys()):
        raise ValueError(
            f"Missing event codes: {event_id_raw.keys()}"
        )

    event_id = {
        "rest": event_id_raw["T0"],
        "left": event_id_raw["T1"],
        "right": event_id_raw["T2"],
    }

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=0.0,
        tmax=4.0,
        baseline=None,
        preload=True,
        event_repeated="drop",
        verbose=False
    )

    return epochs

In [40]:
total = 0
success = 0
failed = []

In [41]:
for subject in sorted(os.listdir(FILTERED_ROOT)):

    subject_dir = os.path.join(
        FILTERED_ROOT,
        subject
    )

    if not os.path.isdir(subject_dir):
        continue

    print("\n" + "=" * 60)
    print(subject)
    print("=" * 60)

    output_dir = os.path.join(
        EPOCH_ROOT,
        subject
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    for file in sorted(os.listdir(subject_dir)):

        if not file.endswith(".fif"):
            continue

        if not any(run in file for run in TARGET_RUNS):
            continue

        total += 1

        input_path = os.path.join(
            subject_dir,
            file
        )

        output_name = file.replace(
            "_filtered_raw.fif",
            "_epo.fif"
        )

        output_path = os.path.join(
            output_dir,
            output_name
        )

        if os.path.exists(output_path):
            print(f"✓ Exists: {output_name}")
            continue

        try:

            print(f"Epoching {file}")

            raw = mne.io.read_raw_fif(
                input_path,
                preload=True,
                verbose=False
            )

            epochs = create_epochs_from_filtered(raw)

            epochs.save(
                output_path,
                overwrite=True,
                verbose=False
            )

            success += 1

        except Exception as e:

            failed.append(
                (subject, file, str(e))
            )

            print(f"FAILED: {e}")


S001
Epoching S001R04_filtered_raw.fif
Epoching S001R08_filtered_raw.fif
Epoching S001R12_filtered_raw.fif

S002
Epoching S002R04_filtered_raw.fif
Epoching S002R08_filtered_raw.fif
Epoching S002R12_filtered_raw.fif

S003
Epoching S003R04_filtered_raw.fif
Epoching S003R08_filtered_raw.fif
Epoching S003R12_filtered_raw.fif

S004
Epoching S004R04_filtered_raw.fif
Epoching S004R08_filtered_raw.fif
Epoching S004R12_filtered_raw.fif

S005
Epoching S005R04_filtered_raw.fif
Epoching S005R08_filtered_raw.fif
Epoching S005R12_filtered_raw.fif

S006
Epoching S006R04_filtered_raw.fif
Epoching S006R08_filtered_raw.fif
Epoching S006R12_filtered_raw.fif

S007
Epoching S007R04_filtered_raw.fif
Epoching S007R08_filtered_raw.fif
Epoching S007R12_filtered_raw.fif

S008
Epoching S008R04_filtered_raw.fif
Epoching S008R08_filtered_raw.fif
Epoching S008R12_filtered_raw.fif

S009
Epoching S009R04_filtered_raw.fif
Epoching S009R08_filtered_raw.fif
Epoching S009R12_filtered_raw.fif

S010
Epoching S010R04_filte

In [42]:
print("\n")
print("=" * 60)
print("EPOCHING COMPLETE")
print("=" * 60)

print("Total files :", total)
print("Success     :", success)
print("Failed      :", len(failed))

for item in failed:
    print(item)



EPOCHING COMPLETE
Total files : 327
Success     : 327
Failed      : 0
